# MongoDB config servers — infrastructure

This notebook deploys **only the config servers** (`configsvr` role) that form the `config_rs` replica set.

## 1. Parameters

- **What:** default values for `NETWORK_MODE`, the VPN identity and `START_FROM_SCRATCH`.
- **Why:** the same notebook serves Local mode (ports on `127.0.0.1`) and VPN mode (`<container>.<alias>.<domain>`).
- **How:** the cell is tagged `parameters`; papermill inserts an `injected-parameters` cell right after it with the `-p` values.
- 📄 [papermill — Parameterize](https://papermill.readthedocs.io/en/latest/usage-parameterize.html)

In [ ]:
# Local mode is the portable default; VPN mode publishes services on this host.
NETWORK_MODE = "local"
VPN_HOST_IP = "10.15.20.100"
VPN_DNS_IP = "10.15.20.1"
VPN_DOMAIN = "vpn.itam.mx"
VPN_CLIENT_ALIAS = "mavasbel"
START_FROM_SCRATCH = False

## 2. Validate the parameters

- **What:** lower-cases `NETWORK_MODE` and `VPN_CLIENT_ALIAS` and builds `VPN_CLIENT_DOMAIN = <alias>.<domain>`.
- **Why:** a wrong value must stop the notebook here, before any container changes.
- **How:** rejects any mode other than `local`/`vpn` and, in VPN mode, a host IP outside `10.15.20.*`, and an alias that is not lower-case letters, digits and inner hyphens.

In [ ]:
import re

NETWORK_MODE = NETWORK_MODE.strip().lower()
if NETWORK_MODE not in {"local", "vpn"}:
    raise ValueError("NETWORK_MODE must be 'local' or 'vpn'")
if NETWORK_MODE == "vpn" and not VPN_HOST_IP.startswith("10.15.20."):
    raise ValueError("VPN_HOST_IP must be in the 10.15.20.* subnet")
VPN_CLIENT_ALIAS = VPN_CLIENT_ALIAS.strip().lower()
if not VPN_CLIENT_ALIAS:
    raise ValueError("VPN_CLIENT_ALIAS must not be empty")
if re.fullmatch(r"[a-z0-9](?:[a-z0-9-]*[a-z0-9])?", VPN_CLIENT_ALIAS) is None:
    raise ValueError(
        "VPN_CLIENT_ALIAS must contain only lowercase letters, digits, or hyphens "
        "and must not start or end with a hyphen"
    )
VPN_CLIENT_DOMAIN = f"{VPN_CLIENT_ALIAS}.{VPN_DOMAIN}"

## 3. Find the module folder

- **What:** locates the labs-setup root, changes into `mongodb/` and imports `run_command`.
- **Why:** relative paths (Compose file, `mount/`) then resolve the same way whether papermill or JupyterLab started the kernel; `run_command` passes an argument list to the program without a shell, so it behaves the same on Windows, macOS and Linux.
- **How:** walks up from the current folder to the first one holding `pyproject.toml`, `cassandra/` and `mongodb/`.

In [ ]:
# Resolve module assets from the labs-setup root.
import os
from pathlib import Path

_start = Path.cwd().resolve()
LABS_ROOT = next(
    (
        candidate
        for candidate in (_start, *_start.parents)
        if (candidate / "pyproject.toml").is_file()
        and (candidate / "cassandra").is_dir()
        and (candidate / "mongodb").is_dir()
    ),
    None,
)
if LABS_ROOT is None:
    raise RuntimeError(
        "Could not find the labs-setup root from the current directory"
    )
MODULE_DIR = LABS_ROOT / "mongodb"
import sys

if str(LABS_ROOT) not in sys.path:
    sys.path.insert(0, str(LABS_ROOT))
from labs_runtime import run_command

os.chdir(MODULE_DIR)
print(f"Working directory: {MODULE_DIR}")

## 4. Config server settings and network

- **What:** names three servers `config-server-1..3` on ports `27111-27113` with root user `admin`/`admin`, and makes sure the Docker network `mongodb-network` exists.
- **Why:** config servers hold the metadata of a sharded cluster and must run as a replica set; they join the same network as the `mongodb_infra` members, so this notebook creates it if `mongodb_infra` has not run.
- **How:** `docker network inspect` decides whether `docker network create mongodb-network` is needed.
- 📄 [MongoDB 7.0 — Config servers](https://www.mongodb.com/docs/v7.0/core/sharded-cluster-config-servers/) · [docker network create](https://docs.docker.com/reference/cli/docker/network/create/)

In [ ]:
MONGODB_START_FROM_SCRATCH = START_FROM_SCRATCH
DOCKER_INTERNAL_HOST = "host.docker.internal"
DOCKER_DNS = [VPN_DNS_IP] if NETWORK_MODE == "vpn" else []
HOST_BIND_IP = VPN_HOST_IP if NETWORK_MODE == "vpn" else "127.0.0.1"

# --- CONFIG SERVER CONFIGURATION ---
MONGODB_CONFIG_SVR_NODES = 3
MONGODB_STARTING_PORT = 27110
# ------------------------------------

MONGO_INITDB_ROOT_USERNAME = "admin"
MONGO_INITDB_ROOT_PASSWORD = "admin"
MONGO_INITDB_DATABASE = "admin"

# --- DYNAMIC NAME & PORT GENERATION ---
MONGODB_CONFIG_SVR_NAMES = [
    f"config-server-{i+1}" for i in range(MONGODB_CONFIG_SVR_NODES)
]
MONGODB_CONFIG_SVR_PORTS = [
    MONGODB_STARTING_PORT + (i + 1) for i in range(MONGODB_CONFIG_SVR_NODES)
]
def vpn_fqdn(container_name):
    return f"{container_name}.{VPN_CLIENT_DOMAIN}"


MONGODB_CONFIG_SVR_INTERNAL_HOSTS = MONGODB_CONFIG_SVR_NAMES
MONGODB_CONFIG_SVR_CLIENT_HOSTS = [
    "127.0.0.1" if NETWORK_MODE == "local" else vpn_fqdn(name)
    for name in MONGODB_CONFIG_SVR_NAMES
]
MONGODB_CONFIG_SVR_ADVERTISED_HOSTS = [
    name if NETWORK_MODE == "local" else vpn_fqdn(name)
    for name in MONGODB_CONFIG_SVR_NAMES
]
MONGODB_CONFIG_SVR_COMPOSE_HOSTS = [
    name if NETWORK_MODE == "local" else vpn_fqdn(name)
    for name in MONGODB_CONFIG_SVR_NAMES
]

print(
    "Config servers:",
    *list(
        zip(
            MONGODB_CONFIG_SVR_NAMES,
            MONGODB_CONFIG_SVR_CLIENT_HOSTS,
            MONGODB_CONFIG_SVR_PORTS,
        )
    ),
    sep="\n",
)

import subprocess

network_exists = subprocess.run(
    ["docker", "network", "inspect", "mongodb-network"],
    capture_output=True,
    text=True,
    encoding="utf-8",
).returncode == 0
if not network_exists:
    subprocess.run(
        ["docker", "network", "create", "mongodb-network"], check=True
    )

## 5. Host folders

- **What:** defines `config/` (generated files), `configsvr-mount/` (data and config per server) and the keyfile path, and creates `configsvr-mount/`.
- **Why:** the generated files and the data survive container restarts.
- **How:** paths built from `MODULE_DIR` with `os.path.join`.
- 📄 [Docker — Bind mounts](https://docs.docker.com/engine/storage/bind-mounts/)

In [ ]:
import os
from pathlib import Path

LOCALHOST_WORKDIR = str(MODULE_DIR.resolve())
LOCALHOST_CONFIG_DIR = os.path.abspath("config")
LOCALHOST_MOUNTDIR = os.path.join(LOCALHOST_WORKDIR, "configsvr-mount")
MONGODB_LOCAL_CLUSTER_KEY_PATH = os.path.join(LOCALHOST_WORKDIR, "mongo-keyfile")

mount_path = Path(LOCALHOST_MOUNTDIR)
mount_path.mkdir(parents=True, exist_ok=True)

## 6. Stop the previous config servers

- **What:** with `START_FROM_SCRATCH=True`, removes their containers and volumes.
- **Why:** a fresh build must not reuse old servers; with `False` they are kept.
- **How:** `docker compose -f mongodb-configsvr.docker-compose.yml down -v` (`check=False`).
- 📄 [docker compose down](https://docs.docker.com/reference/cli/docker/compose/down/)

In [ ]:
if MONGODB_START_FROM_SCRATCH:
    subprocess.run(
        ["docker", "compose", "-f", "mongodb-configsvr.docker-compose.yml", "down", "-v"],
        check=False,
    )
else:
    print("Preserving existing containers and volumes")

## 7. Delete the old state

- **What:** with `START_FROM_SCRATCH=True`, deletes the keyfile and empties `config/` and `configsvr-mount/`.
- **Why:** files written by the containers belong to uid 999 and the host user cannot always delete them.
- **How:** the keyfile is made writable and removed on the host; the folders are emptied with `rm -rf` as root in a throwaway container.
- 📄 [docker container run](https://docs.docker.com/reference/cli/docker/container/run/)

In [ ]:
import shutil
import stat

def clear_bind_directory(path):
    target = Path(path).resolve()
    target.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [
            "docker",
            "run",
            "--rm",
            "--mount",
            f"type=bind,source={target},target=/target",
            "mongo:8.2.3",
            "bash",
            "-c",
            "find /target -mindepth 1 -maxdepth 1 -exec rm -rf -- {} +",
        ],
        check=True,
    )

if MONGODB_START_FROM_SCRATCH:
    if os.path.exists(MONGODB_LOCAL_CLUSTER_KEY_PATH):
        os.chmod(MONGODB_LOCAL_CLUSTER_KEY_PATH, stat.S_IWRITE)
        os.remove(MONGODB_LOCAL_CLUSTER_KEY_PATH)
    clear_bind_directory(LOCALHOST_CONFIG_DIR)
    clear_bind_directory(LOCALHOST_MOUNTDIR)

## 8. Keyfile

- **What:** writes a random keyfile (756 bytes, base64) unless one exists.
- **Why:** the servers authenticate to each other with it, which also turns on access control.
- **How:** `secrets.token_bytes(756)` encoded in base64.
- 📄 [Internal/membership authentication](https://www.mongodb.com/docs/v7.0/core/security-internal-authentication/)

In [ ]:
import yaml
import base64
import secrets
from IPython.display import Markdown, display

# --- Generate keyfile if it doesn't exist ---
if not os.path.exists(MONGODB_LOCAL_CLUSTER_KEY_PATH):
    with open(MONGODB_LOCAL_CLUSTER_KEY_PATH, "w", encoding="utf-8", newline="\n") as f:
        raw_data = secrets.token_bytes(756)
        f.write(base64.b64encode(raw_data).decode("utf-8"))

## 9. mongod.conf and bootstrap script

- **What:** writes, per server, a `mongod.conf` (`clusterRole: configsvr`, `replSetName: config_rs`, keyfile, its port) and `mongod-bootstrap.sh`.
- **Why:** the services start a bash script instead of `mongod`, and the image's entrypoint only creates the root user from `MONGO_INITDB_ROOT_*` when the command starts with `mongod`; so the script creates it itself.
- **How:** on an empty data folder the script starts a temporary `mongod` without keyfile or replica set, bound to `127.0.0.1` so nothing outside the container can connect while it has no access control; it runs `db.createUser` with role `root`, stops it, then `exec`s the real `mongod --config`. On a used data folder it starts the real one directly.
- 📄 [Configuration file options](https://www.mongodb.com/docs/v7.0/reference/configuration-options/) · [db.createUser](https://www.mongodb.com/docs/v7.0/reference/method/db.createUser/) · [replSetInitiate](https://www.mongodb.com/docs/v7.0/reference/command/replSetInitiate/)

In [ ]:
# --- Generate mongod.conf and mongod-bootstrap.sh for each config server ---
#
# mongod-bootstrap.sh implements the two-phase startup that docker-entrypoint.sh
# cannot handle for configsvr nodes:
#   Phase 1 – plain standalone (no configsvr/keyFile/replSet) to create the admin user
#   Phase 2 – exec real mongod with the full config

MONGO_REPLICA_SET_NAME = "config_rs"
MONGO_BOOTSTRAP_SCRIPT = """\
#!/bin/bash
set -Eeuo pipefail

# --- Step 1: Secure the keyfile ---
cp -f /etc/mongo/config/mongo-keyfile /data/keyfile
chmod 400 /data/keyfile
chown 999:999 /data/keyfile

# --- Step 2: Skip init if already initialized ---
for marker in /data/db/WiredTiger /data/db/storage.bson /data/db/journal; do
    if [ -e "$marker" ]; then
        echo "[bootstrap] Data directory already initialized – skipping user creation."
        exec mongod --config /etc/mongo/config/mongod.conf
    fi
done

# --- Step 3: Phase 1 – plain standalone, no configsvr/keyFile/replSet ---
echo "[bootstrap] Starting temporary standalone mongod for user initialisation..."
mongod \\
    --dbpath /data/db \\
    --bind_ip 127.0.0.1 \\
    --port 27017 \\
    --fork \\
    --logpath /tmp/mongod-init.log

echo "[bootstrap] Waiting for temporary mongod to accept connections..."
until mongosh --quiet --eval "db.adminCommand('ping')" > /dev/null 2>&1; do
    sleep 1
done

echo "[bootstrap] Creating admin user..."
mongosh admin --quiet --eval "
db.createUser({
    user: '$MONGO_INITDB_ROOT_USERNAME',
    pwd:  '$MONGO_INITDB_ROOT_PASSWORD',
    roles: [{ role: 'root', db: '$MONGO_INITDB_DATABASE' }]
});
"

echo "[bootstrap] Shutting down temporary mongod..."
mongod --dbpath /data/db --shutdown

echo "[bootstrap] Init complete – starting production mongod."

# --- Step 4: Phase 2 – real mongod with keyFile, configsvr, replSet ---
exec mongod --config /etc/mongo/config/mongod.conf
"""

os.makedirs(LOCALHOST_CONFIG_DIR, exist_ok=True)
for i, node_name in enumerate(MONGODB_CONFIG_SVR_NAMES):
    node_config_dir = os.path.join(LOCALHOST_CONFIG_DIR, node_name)
    os.makedirs(node_config_dir, exist_ok=True)

    # Keyfile
    shutil.copy(
        MONGODB_LOCAL_CLUSTER_KEY_PATH, os.path.join(node_config_dir, "mongo-keyfile")
    )

    # mongod.conf — storage.dbPath explicit so bootstrap finds the right directory
    config_dict = {
        "storage": {"dbPath": "/data/db"},
        "net": {"bindIp": "0.0.0.0", "port": MONGODB_CONFIG_SVR_PORTS[i]},
        "security": {"keyFile": "/data/keyfile"},
        "sharding": {"clusterRole": "configsvr"},
        "replication": {"replSetName": MONGO_REPLICA_SET_NAME},
    }
    with open(
        os.path.join(node_config_dir, "mongod.conf"),
        "w",
        encoding="utf-8",
        newline="\n",
    ) as f:
        yaml.dump(config_dict, f, default_flow_style=False)

    # mongod-bootstrap.sh
    bootstrap_path = os.path.join(node_config_dir, "mongod-bootstrap.sh")
    with open(bootstrap_path, "w", encoding="utf-8", newline="\n") as f:
        f.write(MONGO_BOOTSTRAP_SCRIPT)

print("✅ Configuration files generated in 'config/' directory.")

## 10. Copy the files into the mounts

- **What:** copies each server's `config/<server>/` into `configsvr-mount/<server>/config` and creates its `data` folder.
- **Why:** each container mounts only its own folders.
- **How:** `shutil.copytree(..., dirs_exist_ok=True)`; with `START_FROM_SCRATCH=True` old copies are deleted first.
- 📄 [Docker — Bind mounts](https://docs.docker.com/engine/storage/bind-mounts/)

In [ ]:
# --- Copy config files into mount directories ---
for node_name in MONGODB_CONFIG_SVR_NAMES:
    src_config_dir = os.path.join(LOCALHOST_CONFIG_DIR, node_name)
    dest_config_dir = os.path.join(LOCALHOST_MOUNTDIR, node_name, "config")
    dest_data_dir = os.path.join(LOCALHOST_MOUNTDIR, node_name, "data")

    if MONGODB_START_FROM_SCRATCH and os.path.exists(dest_config_dir):
        shutil.rmtree(dest_config_dir)
    if MONGODB_START_FROM_SCRATCH and os.path.exists(dest_data_dir):
        shutil.rmtree(dest_data_dir)
    os.makedirs(dest_config_dir, exist_ok=True)
    os.makedirs(dest_data_dir, exist_ok=True)

    shutil.copytree(src_config_dir, dest_config_dir, dirs_exist_ok=True)

print("✅ Configuration copied to mount directories.")

## 11. Write the Compose file

- **What:** generates `mongodb-configsvr.docker-compose.yml`: three `mongo:7.0` services running the bootstrap script, 1 CPU and 512 MB each.
- **Why:** the file is generated, never edited by hand.
- **How:** the services join the existing network (`external: true`); the health check is `db.adminCommand('ping')`; server N starts after server N-1.
- 📄 [Compose file — networks (`external`)](https://docs.docker.com/reference/compose-file/networks/) · [Compose file — services](https://docs.docker.com/reference/compose-file/services/)

In [ ]:
# --- Generate Docker Compose file ---
configsvr_compose_dict = {
    "name": "mongodb-configsvr",
    "services": {},
    "networks": {"mongodb-network": {"external": True, "name": "mongodb-network"}},
}

# The bootstrap script handles keyfile copy + two-phase init + real mongod exec.
# MONGO_INITDB_ROOT_* env vars are read directly by the script via shell expansion.
cmd_script = "bash /etc/mongo/config/mongod-bootstrap.sh"

for i, node_name in enumerate(MONGODB_CONFIG_SVR_NAMES):
    hostname = MONGODB_CONFIG_SVR_COMPOSE_HOSTS[i]
    port = MONGODB_CONFIG_SVR_PORTS[i]

    service_def = {
        "image": "mongo:7.0",
        "container_name": node_name,
        "hostname": hostname,
        "command": ["bash", "-c", cmd_script],
        "environment": [
            f"MONGO_INITDB_ROOT_USERNAME={MONGO_INITDB_ROOT_USERNAME}",
            f"MONGO_INITDB_ROOT_PASSWORD={MONGO_INITDB_ROOT_PASSWORD}",
            f"MONGO_INITDB_DATABASE={MONGO_INITDB_DATABASE}",
        ],
        "volumes": [
            f"{os.path.join(LOCALHOST_MOUNTDIR, node_name, 'data')}:/data/db",
            f"{os.path.join(LOCALHOST_MOUNTDIR, node_name, 'config')}:/etc/mongo/config",
        ],
        "networks": ["mongodb-network"],
        "ports": [f"{HOST_BIND_IP}:{port}:{port}"],
        "extra_hosts": [f"{DOCKER_INTERNAL_HOST}:host-gateway"],
        "dns": DOCKER_DNS,
        "deploy": {"resources": {"limits": {"cpus": "1.0", "memory": "512M"}}},
        "healthcheck": {
            "test": [
                "CMD",
                "mongosh",
                "--port",
                f"{port}",
                "--quiet",
                "--eval",
                "db.adminCommand('ping')",
            ],
            "interval": "10s",
            "timeout": "10s",
            "retries": 10,
            "start_period": "30s",  # allow time for the two-phase init
        },
        "depends_on": {},
    }

    # Each server depends on the previous one having started
    if i > 0:
        service_def["depends_on"][MONGODB_CONFIG_SVR_NAMES[i - 1]] = {
            "condition": "service_started"
        }

    configsvr_compose_dict["services"][node_name] = service_def

compose_yaml_path = os.path.join(
    LOCALHOST_WORKDIR, "mongodb-configsvr.docker-compose.yml"
)
with open(compose_yaml_path, "w", encoding="utf-8", newline="\n") as f:
    yaml.dump(
        configsvr_compose_dict, f, default_flow_style=False, sort_keys=False, indent=4
    )

print(f"Successfully created: '{os.path.relpath(compose_yaml_path)}'")
display(
    Markdown(
        f"```yaml\n{yaml.dump(configsvr_compose_dict, default_flow_style=False, sort_keys=False, indent=4)}\n```"
    )
)

## 12. Start the config servers

- **What:** starts the three servers and waits until their health checks pass.
- **Why:** the replica set can only be initiated once every server answers.
- **How:** `docker compose up -d --wait`.
- 📄 [docker compose up](https://docs.docker.com/reference/cli/docker/compose/up/)

In [ ]:
run_command(["docker", "compose", "-f", "mongodb-configsvr.docker-compose.yml", "up", "-d", "--wait"])

## 13. Initiate `config_rs`

- **What:** waits until all three servers answer `ping` three times in a row (up to 120 s), then sends `replSetInitiate` with `configsvr: true`.
- **Why:** the configuration lists all three servers, so each must be up and accept the root login before it is sent.
- **How:** direct connections with the root user; members are announced by container name (Local) or by VPN name (VPN); an "already initialized" error is accepted on a re-run.
- 📄 [Replica set configuration](https://www.mongodb.com/docs/v7.0/reference/replica-configuration/) · [replSetInitiate](https://www.mongodb.com/docs/v7.0/reference/command/replSetInitiate/) · [PyMongo 4.17.0 — MongoClient](https://pymongo.readthedocs.io/en/4.17.0/api/pymongo/mongo_client.html)

In [ ]:
import time
from pymongo import MongoClient
from pymongo.errors import OperationFailure

client_options = {"directConnection": True, "serverSelectionTimeoutMS": 20000}

def config_server_uri(i):
    return (
        f"mongodb://{MONGO_INITDB_ROOT_USERNAME}:{MONGO_INITDB_ROOT_PASSWORD}"
        f"@{MONGODB_CONFIG_SVR_CLIENT_HOSTS[i]}:{MONGODB_CONFIG_SVR_PORTS[i]}"
        "/?authSource=admin"
    )


readiness_deadline = time.time() + 120
consecutive_ready_checks = 0
while time.time() < readiness_deadline and consecutive_ready_checks < 3:
    all_nodes_ready = True
    for i in range(MONGODB_CONFIG_SVR_NODES):
        try:
            with MongoClient(config_server_uri(i), **client_options) as client:
                client.admin.command("ping")
        except Exception:
            all_nodes_ready = False
            break
    consecutive_ready_checks = consecutive_ready_checks + 1 if all_nodes_ready else 0
    if consecutive_ready_checks < 3:
        time.sleep(1)

if consecutive_ready_checks < 3:
    raise TimeoutError("Config servers did not become stably ready within 120s")

auth_uri = config_server_uri(0)

replica_set_config = {
    "_id": MONGO_REPLICA_SET_NAME,
    "configsvr": True,
    "members": [
        {
            "_id": i,
            "host": f"{MONGODB_CONFIG_SVR_ADVERTISED_HOSTS[i]}:{MONGODB_CONFIG_SVR_PORTS[i]}",
        }
        for i in range(MONGODB_CONFIG_SVR_NODES)
    ],
}

print(
    f"🚀 Initiating replica set '{MONGO_REPLICA_SET_NAME}' on "
    f"{MONGODB_CONFIG_SVR_CLIENT_HOSTS[0]}:{MONGODB_CONFIG_SVR_PORTS[0]} ..."
)
for m in replica_set_config["members"]:
    print(f"   _id={m['_id']}  host={m['host']}")

try:
    with MongoClient(auth_uri, **client_options) as client:
        client.admin.command("replSetInitiate", replica_set_config)
        print("✅ replSetInitiate accepted.")
except OperationFailure as e:
    if "already initialized" in str(e).lower():
        print(
            f"⚠️  Replica set '{MONGO_REPLICA_SET_NAME}' is already initiated. Skipping..."
        )
    else:
        raise

## 14. Wait for a primary

- **What:** polls `hello` on each server for up to 30 s until one is primary, then checks 1 `PRIMARY` and 2 `SECONDARY`.
- **Why:** the config replica set is usable only once a primary is elected.
- **How:** `isWritablePrimary` from `hello`, member states from `replSetGetStatus`; a timeout raises an error.
- 📄 [hello](https://www.mongodb.com/docs/v7.0/reference/command/hello/) · [replSetGetStatus](https://www.mongodb.com/docs/v7.0/reference/command/replSetGetStatus/)

In [ ]:
import time

PRIMARY_ELECTION_TIMEOUT = 30

primary = None
primary_index = None
start_time = time.time()
while primary is None and time.time() < start_time + PRIMARY_ELECTION_TIMEOUT:
    for i in range(MONGODB_CONFIG_SVR_NODES):
        try:
            with MongoClient(config_server_uri(i), **client_options) as client:
                hello = client.admin.command("hello")
                if hello.get("isWritablePrimary") or hello.get("ismaster"):
                    primary_index = i
                    status = client.admin.command("replSetGetStatus")
                    primary = next(
                        (
                            member
                            for member in status.get("members", [])
                            if member["stateStr"] == "PRIMARY"
                        ),
                        None,
                    )
                    break
        except Exception:
            continue
    if primary is None:
        print("⏳ No PRIMARY yet, retrying in 1 s...")
        time.sleep(1)

if primary is None:
    raise TimeoutError(f"No PRIMARY elected within {PRIMARY_ELECTION_TIMEOUT}s.")

assert len(status["members"]) == MONGODB_CONFIG_SVR_NODES
assert sum(m["stateStr"] == "PRIMARY" for m in status["members"]) == 1
assert sum(m["stateStr"] == "SECONDARY" for m in status["members"]) == 2
print(
    f"Primary reached at {MONGODB_CONFIG_SVR_CLIENT_HOSTS[primary_index]}:"
    f"{MONGODB_CONFIG_SVR_PORTS[primary_index]}"
)
print(f"Replica Set '{MONGO_REPLICA_SET_NAME}' Status Summary:")
for m in status["members"]:
    icon = "🟢" if m["health"] == 1 else "🔴"
    print(f"{icon} {m['name']:<35} | {m['stateStr']:<10}")